## Part 2: App Information Extraction & Filtering

In [ ]:
import json
import re
import string
import html
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
date_str = "20250904" #"20260518"

In [ ]:
# Load raw app data
load_path = f"../data/raw/medical_health_education_apps_{date_str}_android.json"
with open(load_path, "r", encoding="utf-8") as f:
    data = json.load(f)
print(f"Length of data: {len(data)}")

In [ ]:
# Helper: extract the latest review date from an app's reviews list
def get_latest_review_date(reviews):
    """Return the most recent review date as pd.Timestamp, or NaT if unavailable."""
    if not reviews:
        return pd.NaT
    dates = []
    for r in reviews:
        # google-play-scraper uses 'at' as the review timestamp
        d = r.get('at') if isinstance(r, dict) else None
        if d is not None:
            dates.append(pd.to_datetime(d, errors='coerce'))
    dates = [d for d in dates if pd.notna(d)]
    return max(dates) if dates else pd.NaT


# Extract app-level information into DataFrame
df = pd.DataFrame([{
    'AppId':              d['appId'],
    'Title':              str(d['title']),
    'Score':              d['score'],
    'Genre':              d['genre'],
    'Free':               d['free'],
    'Nb_Installs':        d['installs'],
    'Nb_Reviews':         len(d['reviews']),
    'Latest_Review_Date': get_latest_review_date(d['reviews']),
    'Description':        d['description']
} for d in data])
df.info()

In [ ]:
import os
import json as _json
# Load the API key from config_local.json in the parent folder (kept out of the repo)
with open("../../config_local.json", "r", encoding="utf-8") as _f:
    _cfg = _json.load(_f)
os.environ["OPENAI_API_KEY"] = _cfg["openai_api_key"]

# OpenAI client
from openai import OpenAI
client = OpenAI()

import json
import re
import time
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

test_response = client.chat.completions.create(
    model="gpt-5-2025-08-07",
    messages=[
        {"role": "user", "content": "Reply with 'OK' only."}
    ],
)
print(test_response.choices[0].message.content)
print("Model used:", test_response.model)

In [ ]:
import os
import json
import re
import time
import pandas as pd
from tqdm import tqdm
from openai import OpenAI


# ============================================================
# ╔═══════════════════════════════════════════════════════╗
# ║  USER-EDITABLE SECTION                                ║
# ║  Edit only the values in this section.                ║
# ╚═══════════════════════════════════════════════════════╝
# ============================================================

# --- 1. Model & filters --------------------------------------
GPT_MODEL       = "gpt-5-2025-08-07"
RECENCY_CUTOFF  = pd.Timestamp("2025-06-01")
MIN_REVIEWS     = 50
INSTALL_MIN     = 1_000
INSTALL_MAX     = 10_000_000

# --- 2. Output ------------------------------------------------
OUT_APP_CSV = f"../data/processed_{date_str}/app_information_with_description.csv"  # deliverable (app info + GPT draft class.)
OUT_JSONL   = f"../outputs/gpt5_raw_responses_{date_str}.jsonl"           # raw GPT log (audit / resume)

# --- 3. Categories -------------------------------------------
# Classification categories.
# - machine_key: keyword the model emits (parsed from the response)
# - prompt_block: category description injected into the system prompt
# To add / change / remove a category, edit only this dict.
# The parser reads this dict automatically.
CATEGORIES = {
    "consumer": {
        "prompt_block": (
            "1 → Consumer / Patients / Health-conscious Individuals\n"
            "   short label: general users or patients\n"
            "   machine key: consumer"
        ),
    },
    "provider": {
        "prompt_block": (
            "2 → Healthcare Professionals / Medical Students / Provider / "
            "Health-related practitioners and trainees\n"
            "   short label: HCPs / Med Students\n"
            "   machine key: provider"
        ),
    },
    "other_unclear": {
        "prompt_block": (
            "3 → Other / Unclear Audience\n"
            "   short label: Other / Unclear\n"
            "   machine key: other_unclear"
        ),
    },
}

# --- 4. Instruction (guidance excluding the category blocks) -------------
# Prepended above the category blocks.
PROMPT_INSTRUCTION = (
    "In this study, we categorized mobile health applications (mHealth apps) by their primary target audience into two types. Consumer-targeted apps are apps designed for general users or patients, supporting personal health management or self-care. Provider-targeted apps are apps designed for health-related practitioners and trainees, supporting professional practice or specialized learning. The provider category encompasses clinicians, allied health and therapy professionals, wellness and fitness practitioners, and students or trainees in health-related disciplines. We refer to their corresponding review datasets as the consumer-targeted group and provider-targeted group, respectively."
    "이 정의에 맞추어서 consumer인지, provider인지, other(unclassified)인지 분류하고 간단한 이유를 제시해줘."
    "업무 목적이나 건강 관리가 아닌 단순한 시뮬레이션 게임은 other로 분류해."
    "단순 habit tracking이나 일상 기록과 같은 productivity, tool 앱도 other로 분류하되, 건강에 관련된 것은 consumer로 분류해."
)

# --- 5. machine_key -> Target_Users label (ver5 format, for author review) ---
TARGET_USERS_MAP = {
    "consumer":      "1 Patients / Health-conscious Individuals",
    "provider":      "2 Healthcare Professionals / Medical Students",
    "other_unclear": "3 Other / Unclear Audience",
}


# ============================================================
# ╔═══════════════════════════════════════════════════════╗
# ║  INTERNAL — no need to edit                           ║
# ║  Changes to CATEGORIES / PROMPT_INSTRUCTION propagate ║
# ║  automatically below.                                 ║
# ╚═══════════════════════════════════════════════════════╝
# ============================================================

os.makedirs(os.path.dirname(OUT_APP_CSV), exist_ok=True)
os.makedirs(os.path.dirname(OUT_JSONL), exist_ok=True)

# Build system prompt automatically from CATEGORIES
def build_system_prompt() -> str:
    blocks = [cat["prompt_block"] for cat in CATEGORIES.values()]
    keys = list(CATEGORIES.keys())
    json_examples = " 또는\n".join(
        f'{{"machine_key": "{k}"}}' for k in keys
    )
    return (
        PROMPT_INSTRUCTION
        + "\n\n"
        + "\n\n".join(blocks)
        + "\n\n응답 마지막 줄에 반드시 JSON 형식으로 분류 결과를 출력해줘:\n"
        + json_examples
    )

SYSTEM_PROMPT = build_system_prompt()


# Robust extraction: uses CATEGORIES.keys() automatically
def extract_machine_key(text: str) -> str:
    """
    Extract machine_key from model response.
    Strategy:
      1. Find last JSON block matching {"machine_key": "..."} pattern.
      2. Fallback: scan text for any known machine_key.
    Returns "PARSE_FAILED" if both fail.
    """
    valid_keys = set(CATEGORIES.keys())

    # 1) Last JSON block (handles cases where prompt examples appear earlier)
    json_matches = re.findall(
        r'\{[^{}]*"machine_key"\s*:\s*"([^"]+)"[^{}]*\}', text
    )
    for match in reversed(json_matches):
        key = match.strip()
        if key in valid_keys:
            return key

    # 2) Fallback: scan plain text for any machine_key keyword
    text_lower = text.lower()
    for key in valid_keys:
        if key.lower() in text_lower:
            return key

    return "PARSE_FAILED"


# Sanity check: prompt and parser are in sync
def _sanity_check():
    prompt_keys = set(CATEGORIES.keys())
    test_response = ", ".join(
        f'{{"machine_key": "{k}"}}' for k in prompt_keys
    )
    parsed = {extract_machine_key(f"... {ex} ...")
              for ex in [f'{{"machine_key": "{k}"}}' for k in prompt_keys]}
    if parsed != prompt_keys:
        print(f"[WARNING] Parser-prompt mismatch.")
        print(f"  Prompt keys: {prompt_keys}")
        print(f"  Parsed keys: {parsed}")
    else:
        print(f"[OK] Parser is in sync with {len(prompt_keys)} prompt categories.")

_sanity_check()


# Helpers
def parse_install_bucket(s):
    if not isinstance(s, str):
        return None
    cleaned = s.replace(",", "").replace("+", "").strip()
    try:
        return int(cleaned)
    except ValueError:
        return None


def get_latest_review_date(reviews):
    if not reviews:
        return pd.NaT
    dates = [pd.to_datetime(r.get("at"), errors="coerce")
             for r in reviews if isinstance(r, dict) and r.get("at") is not None]
    dates = [d for d in dates if pd.notna(d)]
    return max(dates) if dates else pd.NaT


# Classification call
client = OpenAI()

def classify_one(description: str, app_id: str) -> dict:
    user_msg = f"App description:\n\n{description}"
    response = client.chat.completions.create(
        model=GPT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
    )
    raw = response.choices[0].message.content
    return {
        "appId":       app_id,
        "machine_key": extract_machine_key(raw),
        "raw":         raw,
        "model":       response.model,
    }


# ============================================================
# Step 1: Filter apps
# (assumes `df` already constructed upstream)
# ============================================================
df["Install_Bucket"] = df["Nb_Installs"].apply(parse_install_bucket)

mask = (
    (df["Nb_Reviews"] >= MIN_REVIEWS)
    & (df["Latest_Review_Date"].notna())
    & (df["Latest_Review_Date"] >= RECENCY_CUTOFF)
    & (df["Free"] == True)
    & (df["Install_Bucket"].notna())
    & (df["Install_Bucket"] >= INSTALL_MIN)
    & (df["Install_Bucket"] <= INSTALL_MAX)
)
df_eligible = df.loc[mask].reset_index(drop=True)
print(f"Apps after filtering: {len(df_eligible)}")


# ============================================================
# Step 2: Run classification with checkpointing
# ============================================================
done_ids = set()
if os.path.exists(OUT_JSONL):
    with open(OUT_JSONL, encoding="utf-8") as f:
        for line in f:
            try:
                done_ids.add(json.loads(line)["appId"])
            except Exception:
                pass
    print(f"Resuming: {len(done_ids)} already classified.")

results = []
for _, row in tqdm(df_eligible.iterrows(), total=len(df_eligible), desc="Classifying"):
    app_id = row["AppId"]
    if app_id in done_ids:
        continue
    desc = str(row.get("Description") or "").strip()
    if not desc:
        out = {"appId": app_id, "machine_key": "other_unclear",
               "raw": "", "model": GPT_MODEL}
    else:
        try:
            out = classify_one(desc, app_id)
        except Exception as e:
            print(f"[ERROR] {app_id}: {e}")
            out = {"appId": app_id, "machine_key": "ERROR",
                   "raw": str(e), "model": GPT_MODEL}
    results.append(out)
    with open(OUT_JSONL, "a", encoding="utf-8") as f:
        f.write(json.dumps(out, ensure_ascii=False) + "\n")
    time.sleep(0.3)


# ============================================================
# Step 3: Merge classification into app info -> app_information_with_description.csv
#   This CSV is the AUTOMATED DRAFT. The authors then review every app, and two
#   corresponding authors validate a random sample by consensus, producing
#   app_target_users_validated.csv (the final labels used downstream).
# ============================================================
recs = []
with open(OUT_JSONL, encoding="utf-8") as f:
    for line in f:
        try:
            recs.append(json.loads(line))
        except Exception:
            pass
df_cls = pd.DataFrame(recs).drop_duplicates(subset=["appId"], keep="last")

df_out = df_eligible.merge(
    df_cls[["appId", "machine_key", "raw"]].rename(columns={"appId": "AppId", "raw": "gpt_reason"}),
    on="AppId", how="left",
)
df_out["Target_Users"] = df_out["machine_key"].map(TARGET_USERS_MAP)

out_cols = ["AppId", "Title", "Score", "Genre", "Free", "Nb_Installs", "Nb_Reviews",
            "Description", "machine_key", "Target_Users", "gpt_reason"]
df_out[out_cols].to_csv(OUT_APP_CSV, index=False, encoding="utf-8-sig")

print(f"\nSaved: {OUT_APP_CSV}  ({len(df_out)} apps)")
print("\nTarget_Users distribution:")
print(df_out["Target_Users"].value_counts(dropna=False))

In [ ]:
# Filter: apps with >= 50 reviews
print(f"Apps with >= 50 reviews: {len(df[df['Nb_Reviews'] >= 50])}")
apps_over50reviews = df[df['Nb_Reviews'] >= 50]
apps_over50reviews.to_csv(f"../data/processed_{date_str}/app_information_over50reviews.csv", index=False)

In [ ]:
# QC: score distribution
df2 = apps_over50reviews[apps_over50reviews['Score'].notna()]

plt.figure(figsize=(15,3))
plt.plot(df2['Score'], '.')
plt.grid()
plt.title("Score distribution (apps with >= 50 reviews)")
plt.show()

print(f"Score range: {df2[df2['Score'] > 0]['Score'].min()} - {df2[df2['Score'] > 0]['Score'].max()}")
print(f"Apps with score: {len(df2[df2['Score'] > 0])}")
print(f"Total reviews: {df2['Nb_Reviews'].sum()}")

## Part 2: Review Cleaning & Consumer/Provider Split
*Source: 03_extract_cleaned_review.ipynb*

In [ ]:
# Emoji removal function
def strip_emoji(text):
    if text is None:
        return None
    s = html.unescape(str(text))
    try:
        import regex as re
        s = re.sub(r'\p{Extended_Pictographic}|\uFE0F|\u200D', '', s)
    except Exception:
        import re
        emoji_re = re.compile(
            "["
            "\U0001F600-\U0001F64F"
            "\U0001F300-\U0001F5FF"
            "\U0001F680-\U0001F6FF"
            "\U0001F700-\U0001F77F"
            "\U0001F780-\U0001F7FF"
            "\U0001F800-\U0001F8FF"
            "\U0001F900-\U0001F9FF"
            "\U0001FA00-\U0001FA6F"
            "\U0001FA70-\U0001FAFF"
            "\U00002702-\U000027B0"
            "\U000024C2-\U0001F251"
            "\U0001F1E6-\U0001F1FF"
            "\u200d"
            "\ufe0f"
            "]+", flags=re.UNICODE
        )
        s = emoji_re.sub('', s)
    import re
    s = re.sub(r"[\r\n]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# ASCII-only cleaning function
allowed_chars = string.ascii_letters + string.digits + string.punctuation + " \t\n\r"
ALLOWED = re.compile(f"[^{re.escape(allowed_chars)}]+")

def keep_keyboard_chars(text):
    if not isinstance(text, str):
        return ""
    cleaned = ALLOWED.sub("", text)
    return re.sub(r"\s+", " ", cleaned).strip()

In [ ]:
# Extract reviews (apps with >= 50 reviews only)
cols = ['appId', 'reviewId', 'content', 'score', 'at']
rows = []

for i in tqdm(range(len(data))):
    reviews = data[i].get('reviews', [])
    if len(reviews) < 50:
        continue
    app_id = data[i].get('appId')
    for r in reviews:
        rows.append({
            'appId':    app_id,
            'reviewId': r.get('reviewId'),
            'content':  strip_emoji(r.get('content')),
            'score':    r.get('score'),
            'at':       r.get('at'),
        })

df_per_review = pd.DataFrame.from_records(rows, columns=cols)

# Whitespace cleanup
df_per_review["content"] = (
    df_per_review["content"]
        .fillna("")
        .str.replace(r"[\r\n]+", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
)

# Generate content_clean BEFORE saving
df_per_review["content_clean"] = df_per_review["content"].apply(keep_keyboard_chars)

# Save
df_per_review.to_csv(
    f"../data/processed_{date_str}/cleaned_review.csv",
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL,
    quotechar='"',
    lineterminator="\n"
)
print(f"Saved cleaned_review.csv: {len(df_per_review)} reviews")

In [ ]:
# Load validated target-user labels (GPT draft + author review + 2-author consensus)
df_cat = pd.read_csv(f'../data/processed_{date_str}/app_target_users_validated.csv')

# ⚠️ Check unique values before filtering — replace strings below if needed
print("Target_Users unique values:", df_cat['Target_Users'].unique())

In [ ]:
# Set category labels explicitly (update if unique values differ)
CONSUMER_LABEL  = 'Consumer'   # ← update if needed
PROVIDER_LABEL  = 'Provider'   # ← update if needed
EXCLUDE_LABEL   = 'Unclassified'  # ← update if needed

final_cat = df_cat[df_cat['Target_Users'] != EXCLUDE_LABEL][['AppId', 'Target_Users']].copy()
final_cat['Target_Users'] = final_cat['Target_Users'].replace({
    CONSUMER_LABEL: 'Consumers',
    PROVIDER_LABEL: 'Providers'
})

final_cat.to_csv(f'../data/processed_{date_str}/app_category_final.csv', index=False)
print(f"Categories: {final_cat['Target_Users'].value_counts().to_dict()}")

In [ ]:
# Split reviews by category and save
cat1 = final_cat[final_cat['Target_Users'] == '1 Patients / Health-conscious Individuals'].reset_index(drop=True)
cat2 = final_cat[final_cat['Target_Users'] == '2 Healthcare Professionals / Medical Students'].reset_index(drop=True)

cat1_review = df_per_review[df_per_review['appId'].isin(cat1['AppId'].unique())].reset_index(drop=True)
cat2_review = df_per_review[df_per_review['appId'].isin(cat2['AppId'].unique())].reset_index(drop=True)

# Full reviews
cat1_review.to_csv(f'../data/processed_{date_str}/reviews_of_category1.csv', index=False)
cat2_review.to_csv(f'../data/processed_{date_str}/reviews_of_category2.csv', index=False)

# content_clean + score only (input for 03_sentiment_scoring)
cat1_review[['content_clean', 'score']].to_csv(f'../data/processed_{date_str}/cleaned_reviews_of_category1.csv', index=False)
cat2_review[['content_clean', 'score']].to_csv(f'../data/processed_{date_str}/cleaned_reviews_of_category2.csv', index=False)

print(f"Consumer reviews: {len(cat1_review)}")
print(f"Provider reviews: {len(cat2_review)}")